# Red neuronal desde cero con NumPy — MNIST

Implementamos y entrenamos una red neuronal **completamente a mano**: capa densa, activaciones, función de pérdida y *backpropagation* escritos desde cero con `numpy`, y visualizamos con `matplotlib`. Sin frameworks de deep learning.

**Objetivos**
- Entender qué hace una capa densa por dentro: inicializar los pesos, multiplicar matrices y sumas el bias.
- Implementar y visualizar las activaciones ReLU y Softmax.
- Escribir la función de pérdida *cross entropy* y ver cómo baja durante el entrenamiento.
- Derivar a mano los gradientes, capa por capa, aplicando la regla de la cadena.
- Evaluar el modelo y analizar en qué se equivoca.

**Estructura**
1. Carga y exploración del dataset MNIST.
2. Implementación: capa densa, activaciones y funciones de pérdida desde cero.
3. Entrenamiento: *backpropagation* manual y descenso por gradiente.
4. Evaluación: curvas, métricas y análisis de errores.
5. Conclusiones.


> **Cómo ejecutarlo:** recreá el entorno una vez con `uv sync` y elegí el kernel del venv
> (`.venv`). El dataset se descarga solo la primera vez y queda cacheado en `data/mnist.npz`.

## 1. Carga y exploración

MNIST es un dataset de dígitos escritos a mano: 70000 imágenes de 28x28 píxeles en escala de grises (60.000 de entrenamiento y 10.000 de test), cada una etiquetada con su dígito (0-9).

La red va a recibir cada imagen **aplanada**: la matriz de 28x28 se convierte en un vector de 784 valores (se hace más abajo). Primero, los datos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests

# Semilla global: fija el "azar" para que todas las corridas den lo mismo.
SEED = 42
np.random.seed(SEED)

# Hiperparámetros del entrenamiento, todos en un solo lugar para poder experimentar.
HIDDEN_SIZE = 128       # neuronas de única capa oculta.
BATCH_SIZE = 64         # muestras por actualización de pesos (mini-batch).
LEARNING_RATE = 0.1     # tamaño del paso del descenso por gradiente.
EPOCHS = 10             # pasadas completas sobre el conjunto del entrenamiento.
VAL_SPLIT = 0.1         # fracción del train reservada para validación.

El dataset se descarga con ``requests`` la primera vez y queda guardado en data/mnist.npz. En la próximas ejecuciones se lee directo del disco, sin volver a descargarlo.

In [ ]:
# El notebook vive en notebooks/, así que los datos van a data en la raíz del proyecto.
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "mnist.npz"
URL_MNIST = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"

def descargar_datos(url: str, destino: Path) -> None:
    """Descarga el dataset si todavía no está cacheado en disco."""
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"Dataset ya disponible en {destino} ({destino.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"Descargando {url} ...")
    respuesta = requests.get(url, timeout=120)
    respuesta.raise_for_status()
    destino.write_bytes(respuesta.content)
    print(f"Descarga completa: {destino} ({destino.stat().st_size / 1e6:.1f} MB)")

descargar_datos(URL_MNIST, DATA_FILE)

In [ ]:
datos = np.load(DATA_FILE)
print("Contenido del archivo", sorted(datos.keys()))

x_train_raw = datos["x_train"]  # imágenes de entrenamiento (60000, 28, 28)
y_train = datos["y_train"]      # etiquetas: (60000,)
x_test_raw = datos["x_test"]    # imágenes de test: (10000, 28, 28)
y_test = datos["y_test"]        # etiquetas: (10000,)

print(f"Entrenamiento: {x_train_raw.shape} -> etiquetas {y_train.shape}")
print(f"Test:          {x_test_raw.shape} -> etiquetas {y_test.shape}")
print(f"Valores de píxel: [{x_train_raw.min()}], {x_train_raw.max()}] en uint8")
print(f"Clases posibles: {np.unique(y_train)}")

Veamos algunas imágenes para entender con qué trabaja la red:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap="gray")
    ax.set_title(f"etiqueta: {y_train[i]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Primeras 10 imágenes del set de entrenamiento", fontsize=12)
plt.tight_layout()

Antes de entrenar hacemos tres chequeos y preparativos:

1. **Balance de clases**: que ningún dígito tenga muchos menos ejemplos que otro.
2. **Aplanar**: cada imagen 28×28 pasa a ser un vector de 784 valores, que es lo que espera la capa densa.
3. **Normalizar**: los píxeles van de 0 a 255; los llevamos al rango [0, 1] dividiendo por 255. Con entradas chicas y parejas, el descenso por gradiente converge mucho mejor.

De paso, separamos un 10 % del entrenamiento como **validación**: ejemplos que el modelo nunca
usa para aprender y que nos dicen si está generalizando.

In [ ]:
# Todos los dígitos deberian tener una cantidad similar de ejemplos (~6000 cada uno).
for digito, cantidad in enumerate(np.bincount(y_train, minlength=10)):
    print(f"Dígito {digito}: {cantidad} ejemplos")

In [ ]:
# Aplanar: (N, 28, 28) -> (N, 784) y normalizar a [0, 1].
x_train = x_train_raw.reshape(len(x_train_raw), -1) / 255
x_test = x_test_raw.reshape(len(x_test_raw), -1) / 255

# Partición train/validación. Mezclamos con la semilla fija para que sea reproducible.
indices = np.random.permutation(len(x_train))
n_val = int(len(x_train) * VAL_SPLIT)
indices_val, indices_train = indices[:n_val], indices[n_val:]

x_val, y_val = x_train[indices_val], y_train[indices_val]
x_train, y_train = x_train[indices_train], y_train[indices_train]

print(f"Train:      {x_train.shape}   ->  (784 = 28x28 píxeles aplanados)")
print(f"Validación: {x_val.shape}")
print(f"Test:       {x_test.shape}")

## 2. Implementación: capa densa, activaciones y pérdida desde cero

### La capa densa

Una capa densa calcula:

$$Z = X W + b$$

donde $X$ tiene forma $(batch, n_{in})$ y $W$ es la matriz de pesos $(n_{in}, n_{out})$.
El producto matricial mezcla **todas** las entradas con **todas** las salidas, y el bias se suma
una vez por neurona. La inicialización (He) escala los pesos con $\sqrt{2 / n_{in}}$ para que
las activaciones no se apaguen ni exploten al propagarse.

En el paso hacia atrás, la regla de la cadena nos da las derivadas que necesitamos:

$$\frac{\partial L}{\partial W} = X^\top \frac{\partial L}{\partial Z}, \qquad
\frac{\partial L}{\partial b} = \sum_{\text{filas}} \frac{\partial L}{\partial Z}, \qquad
\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z} W^\top$$

Las dos primeras sirven para actualizar los parámetros; la tercera es el gradiente que recibe la
capa anterior para seguir propagando hacia atrás.

In [ ]:
class DenseLayer:
    """Capa densa: Z = X @ W + b, con inicialización He y gradientes guardados."""

    def __init__(self, n_in: int, n_out: int) -> None:
        # Inicialización He: Pensada para activaciones ReLU (varianza 2 / n_in).
        self.W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.b = np.zeros(n_out)
        # Gradientes: se recalculan en cada backward
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self._x = None # caché de la última entrada: backward la necesita

    def forward(self, x: np.ndarray) -> np.ndarray:
        self._x = x
        return x @ self.W + self.b

    def backward(self, dz: np.ndarray) -> np.ndarray:
        # dz es el gradiente de la pérdida respecto de la salida de esta capa.
        self.dW = self._x.T @ dz    # (n_in, batch) @ (batch, n_out)
        self.db = dz.sum(axis=0)    # suma sobre el batch
        return dz @ self.W.T        # gradiente respecto de la entrada

### ReLU

$$\text{ReLU}(z) = \max(0, z) \qquad\qquad
\text{ReLU}'(z) = \begin{cases} 1 & \text{si } z > 0 \\ 0 & \text{si } z \le 0 \end{cases}$$

Deja pasar los valores positivos tal cual y apaga los negativos. ¿Por qué necesitamos una
no linealidad? Sin ella, apilar capas sería una única transformación lineal: la red no podría
aprender casi nada. La derivada "binaria" de ReLU hace además que el gradiente fluya sin
encogerse, a diferencia de lo que ocurría con sigmoid o tanh.

In [ ]:
def relu(z: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, z)

def relu_derivada(z: np.ndarray) -> np.ndarray:
    return (z > 0).astype(z.dtype)

In [ ]:
z = np.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(z, relu(z), color="tab:blue")
axes[0].set_title("ReLU: max(0, z)")
axes[0].set_xlabel("z")
axes[0].set_ylabel("ReLU(z)")

axes[1].plot(z, relu_derivada(z), color="tab:orange")
axes[1].set_title("Derivada de ReLU: 1 si z > 0, si no 0")
axes[1].set_xlabel("z")
axes[1].set_ylabel("Relu'(z)")

plt.tight_layout()
plt.show()

### Softmax

Softmax transforma los *logits* del modelo (números reales cualesquiera) en una **distribución
de probabilidad** sobre las 10 clases:

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Cada salida queda entre 0 y 1 y todas suman 1: se puede leer como "qué probabilidad le doy a
cada dígito". Antes de exponenciar restamos el máximo del vector: la cuenta es matemáticamente
idéntica, pero evita el desborde numérico de `exp()` cuando los logits son grandes.

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    # Restar el máximo por fila no cambia el resultado y evita overflow en exp()
    z_estable = z - z.max(axis=1, keepdims=True)
    exponenciales = np.exp(z_estable)
    return exponenciales / exponenciales.sum(axis=1, keepdims=True)

In [ ]:
logits = np.array([[2.0, 1.0, 0.1, -1.0, 3.0]])
#logits = np.array([[2.0, 1.0, 0.1, -1.0, 3.0]]) # Un ejemplo con 5 "clases"

probabilidades = softmax(logits)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(range(1,6), logits[0], color="tab:gray")
axes[0].set_title("Logits: la salida cruda del modelo")
axes[0].set_xlabel("Clase")
axes[0].set_ylabel("valor")

axes[1].bar(range(1,6), probabilidades[0], color="tab:green")
axes[1].set_title("Después de softmax: probabilidades")
axes[1].set_xlabel("clase")
axes[1].set_xlabel("Valor")

plt.tight_layout()
plt.show()

print(f"La suma de probabilidades da {probabilidades.sum():.4f} (siempre 1).")

### Cross-entropy

Para medir el error usamos la *cross-entropy* sobre la clase correcta:

$$L = -\frac{1}{N} \sum_i \log(p_{i,\, y_i})$$

donde $p_{i, y_i}$ es la probabilidad que el modelo le asignó a la clase verdadera de la muestra
$i$. Penaliza fuerte estar *seguro y equivocado* (cuando $p \to 0$, $\log p \to -\infty$) y casi
no castiga estar seguro y acertado.

Hay un detalle elegante: combinada con softmax, la derivada respecto de los logits se simplifica
muchísimo:

$$\frac{\partial L}{\partial Z} = \frac{1}{N} \big(P - Y\big)$$

donde $Y$ es la etiqueta en formato *one-hot*. Esa expresión tan simple es el punto de partida
del *backpropagation*: todo lo demás es aplicar la regla de la cadena hacia atrás.

In [ ]:
def one_hot(y: np.ndarray, n_clases: int = 10) -> np.ndarray:
    """Convierte etiquetas (N,) en una matriz one-hot (N, n_clases)."""
    matriz = np.zeros((len(y), n_clases))
    matriz[np.arange(len(y)), y] = 1.0
    return matriz

def cross_entropy(probs: np.ndarray, y: np.ndarray) -> float:
    """Pérdida promedio: -log de la probabilidad asignada a la clase correcta."""
    n = len(y)
    probabilidad_correcta = probs[np.arange(n), y]
    # El clip evita log(0), que daría infinito si el modelo asigna probabilidad 0
    # a la clase correcta; el max(0) evita el "-0.0" cuando acierta con seguridad.
    return float(max(0.0, -np.log(np.clip(probabilidad_correcta, 1e-12, 1.0)).mean()))

In [ ]:
# Chequeo con casos extremos, para darle sentido a la escala de la pérdida.
acierta_seguro = one_hot(np.array([3]))  # 100 % de probabilidad a la clase correcta
falla_seguro = one_hot(np.array([0]))    # 100 % de probabilidad a una clase incorrecta

print(f"Pérdida si acierta con toda seguridad: {cross_entropy(acierta_seguro, np.array([3])):.4f}")
print(f"Pérdida si falla con toda seguridad:   {cross_entropy(falla_seguro, np.array([3])):.4f}")

## 3. Entrenamiento: backpropagation manual

Armamos la red completa: **784 → 128 (ReLU) → 10 (softmax)**.

### Forward

$$Z_1 = X W_1 + b_1, \qquad A_1 = \text{ReLU}(Z_1), \qquad Z_2 = A_1 W_2 + b_2, \qquad P = \text{softmax}(Z_2)$$

### Backward: la regla de la cadena, de atrás hacia adelante

El gradiente de la pérdida respecto de la salida vale $\frac{\partial L}{\partial Z_2} = \frac{P - Y}{N}$ (lo dedujimos arriba). A partir de ahí:

$$\frac{\partial L}{\partial W_2} = A_1^\top \frac{\partial L}{\partial Z_2}
\qquad
\frac{\partial L}{\partial b_2} = \sum \frac{\partial L}{\partial Z_2}
\qquad
\frac{\partial L}{\partial A_1} = \frac{\partial L}{\partial Z_2} W_2^\top$$

$$\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1} \odot \text{ReLU}'(Z_1)
\qquad
\frac{\partial L}{\partial W_1} = X^\top \frac{\partial L}{\partial Z_1}
\qquad
\frac{\partial L}{\partial b_1} = \sum \frac{\partial L}{\partial Z_1}$$

La idea central: cada capa recibe el gradiente de la pérdida respecto de **su salida** y lo
convierte en dos cosas: el gradiente respecto de **sus parámetros** (para actualizarlos) y el
gradiente respecto de **su entrada** (para seguir hacia atrás). El símbolo $\odot$ es producto
elemento a elemento: la derivada de ReLU decide por qué neuronas "pasa" el gradiente.

### La actualización

Con cada mini-batch de 64 imágenes damos un paso de descenso por gradiente:

$$W \leftarrow W - \eta \, \frac{\partial L}{\partial W}$$

donde $\eta$ es el *learning rate* (0.1). Los mini-batches son un punto medio entre usar una
sola muestra (ruidoso) y el dataset completo (caro): cada paso es una estimación ruidosa del
gradiente real, y ese ruido incluso ayuda a escapar de mínimos malos.

> **Nota práctica:** con 10 epochs, el entrenamiento completo puede tardar unos minutos según la
> máquina. Es el costo de hacer todos los cálculos a mano en NumPy, sin GPU ni kernels optimizados.

In [ ]:
# La red: 784 entradas -> 128 neuronas ocultas -> 10 clases.
capa1 = DenseLayer(784, HIDDEN_SIZE)
capa2 = DenseLayer(HIDDEN_SIZE, 10)

def forward(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pasa x por toda la red; devuelve las probabilidades y Z1 (lo necesita el backward)."""
    z1 = capa1.forward(x)       # (batch, 784) @ (784, 128) -> (batch, 128)
    a1 = relu(z1)               # (batch, 128)
    z2 = capa2.forward(a1)      # (batch, 128) @ (128, 10) -> (batch, 10)
    probs = softmax(z2)         # (batch, 10): cada fila suma 1
    return probs, z1

def backward(probs: np.ndarray, y: np.ndarray, z1: np.ndarray) -> None:
    """Calcula todos los gradientes, de la salida hacia la entrada."""
    n = len(y)
    # Derivada de softmax + cross-entropy (deducida de en la celda de arriba).
    # Ya incluye el promedio 1/N: los gradientes quedan listos para usar
    dz2 = (probs - one_hot(y)) / n
    da1 = capa2.backward(dz2)       # llena dW2 y db2, y devuelve el gradiente de A1
    dz1 = da1 * relu_derivada(z1)   # el gradiente solo pasa donde ReLU está activa
    capa1.backward(dz1)             # llena dW1 y db1

def actualizar_pesos(learning_rate: float) -> None:
    """Un paso de descenso por gradiente sobre los cuatros parámetros."""
    capa1.W -= learning_rate * capa1.dW
    capa1.b -= learning_rate * capa1.db
    capa2.W -= learning_rate * capa2.dW
    capa2.b -= learning_rate * capa2.db

In [ ]:
def evaluar(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Pérdida y exactitud sobre un conjunto completo, sin tocar los pesos."""
    probs, _ = forward(x)
    perdida = cross_entropy(probs, y)
    exactitud = float((probs.argmax(axis=1) == y).mean())
    return perdida, exactitud

In [ ]:
# Métricas por epoch, para graficar más adelante.
historial = {"loss": [], "acc": [], "loss_val": [], "acc_val": []}

for epoch in range(1, EPOCHS + 1):
    # Mezclamos el train en cada epoch: los mini-batches nunca se repiten igual.
    indices = np.random.permutation(len(x_train))

    for inicio in range(0, len(x_train), BATCH_SIZE):
        lote = indices[inicio:inicio + BATCH_SIZE]
        probs, z1 = forward(x_train[lote])   # 1. forward: predecir
        backward(probs, y_train[lote], z1)   # 2. backward: calcular gradientes
        actualizar_pesos(LEARNING_RATE)      # 3. actualizar pesos

    # Al cerrar la epoch, medimos sobre train y validación completos.
    loss_train, acc_train = evaluar(x_train, y_train)
    loss_val, acc_val = evaluar(x_val, y_val)
    historial["loss"].append(loss_train)
    historial["acc"].append(acc_train)
    historial["loss_val"].append(loss_val)
    historial["acc_val"].append(acc_val)
    print(
        f"epoch {epoch:2d}/{EPOCHS}  "
        f"loss train {loss_train:.4f}  acc train {acc_train:.2%}  |  "
        f"loss val {loss_val:.4f}  acc val {acc_val:.2%}"
    )

## 4. Evaluación

### Curvas de entrenamiento

Graficamos la pérdida y la exactitud por epoch, con train y validación juntas. Si las dos bajan,
la red está aprendiendo de verdad. Si la curva de validación se estanca mientras la de train
sigue mejorando, estaríamos frente a *overfitting*: el modelo memoriza el train en lugar de
generalizar.

In [ ]:
epocas = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epocas, historial["loss"], marker="o", label="train")
axes[0].plot(epocas, historial["loss_val"], marker="o", label="validación")
axes[0].set_title("Pérdida (cross-entropy) por epoch")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("pérdida")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epocas, historial["acc"], marker="o", label="train")
axes[1].plot(epocas, historial["acc_val"], marker="o", label="validación")
axes[1].set_title("Exactitud por epoch")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("exactitud")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Evaluación final en test

El conjunto de test no se usó ni para entrenar ni para validar: es la medida honesta de cuánto
generaliza el modelo a imágenes que nunca vio.

In [ ]:
probs_test, _ = forward(x_test)
pred_test = probs_test.argmax(axis=1)

loss_test = cross_entropy(probs_test, y_test)
acc_test = float((pred_test == y_test).mean())

print(f"Pérdida en test:   {loss_test:.4f}")
print(f"Exactitud en test: {acc_test:.2%} ({int(acc_test * len(y_test))} de {len(y_test)} imágenes)")

### ¿Qué dígitos confunde?

La matriz de confusión muestra, para cada dígito real (fila), con qué lo predijo la red
(columna). La diagonal son los aciertos; los errores viven fuera de la diagonal.

In [ ]:
# Matriz de confusión 10x10 filas=etiqueta real, columnas=predicción

confusion = np.zeros((10,10), dtype=int)
np.add.at(confusion, (y_test, pred_test), 1)

# Los errores más frecuentes: miramos fuera de la diagonal.
sin_diagonal = confusion.copy()
np.fill_diagonal(sin_diagonal, 0)

print("="*50)
print("Pares más confundidos (real -> predicho):")
top = np.argsort(sin_diagonal, axis=None)[::-1][:5]
for k in top:
    real, predicho = np.unravel_index(k, sin_diagonal.shape)
    print(f"    real {real} -> predicho {predicho} : {sin_diagonal[real, predicho]} veces")


In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
imagen = ax.imshow(confusion, cmap="Blues")
for i in range(10):
    for j in range(10):
        color = "white" if confusion[i,j] > confusion.max() / 2 else "black"
        ax.text(j, i, confusion[i,j], ha="center", va="center", fontsize=8, color=color)

ax.set_title("Matriz de confusión (test)")
ax.set_xlabel("predicción")
ax.set_ylabel("etiqueta real")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
fig.colorbar(imagen, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Por qué se confunde lo que se confunde

Con la semilla fija de este notebook, la red alcanza ≈ 97,4 % de exactitud en el test y el par
más confundido es **4 → 9**: comparten el trazo superior cerrado y una línea vertical hacia
abajo, y según cómo se escriba el 4 (abierto o cerrado arriba) la diferencia con un 9 queda en
un detalle fino que la red —al igual que un humano apurado— deja pasar. Algo parecido ocurre
con el **7**, que según su inclinación se confunde con el 9 o con el 1; también aparecen cruces
entre 2 y 8, y entre 5 y 3.

Hay dos limitaciones estructurales que explican por qué la red no es perfecta:

- **Sin estructura espacial**: al aplanar la imagen perdemos la noción de píxeles vecinos; la
  red no "ve" trazos, solo posiciones sueltas. Las redes convolucionales existen justamente
  para resolver esto.
- **Capacidad limitada**: una sola capa oculta de 128 neuronas separa bien los dígitos fáciles,
  pero no distingue los casos ambiguos que también engañan al ojo humano.

### Ejemplos concretos

Veamos algunos aciertos y algunos errores. Para los errores elegimos los casos donde la red
estaba **más segura** de su predicción: los más instructivos.

In [ ]:
def mostrar_digitos(imagenes, reales, predichas, indices, titulo):
    """Dibuja una fila de imágenes con su etiqueta real y su predicción."""
    fig, axes = plt.subplots(1, len(indices), figsize=(1.6 * len(indices), 2.4))
    for ax, idx in zip(axes, indices):
        ax.imshow(imagenes[idx], cmap="gray")
        ax.set_title(f"real: {reales[idx]}\npred: {predichas[idx]}", fontsize=9)
        ax.axis("off")
    fig.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    plt.show()

# Para ver las imágenes tal cual son, usamos las originales sin aplanar. (28x28)
aciertos = np.where(pred_test == y_test)[0]
errores = np.where(pred_test != y_test)[0]

rng = np.random.default_rng(SEED)
mostrar_digitos(x_test_raw, y_test, pred_test, rng.choice(aciertos, 8, replace=False), "Ocho aciertos")

# Para los errores, ordenamos por confianza de la predicción y mostramos los 8 peores
confianza = probs_test[errores].max(axis=1)
errores_seguros = errores[np.argsort(confianza)[::-1][:8]]
mostrar_digitos(x_test_raw, y_test, pred_test, errores_seguros, "Ocho errores con alta confianza")


## 5. Conclusiones

Con la red implementada a mano y los hiperparámetros utilizados (batch de 64, learning rate
0.1, 10 épocas), el modelo alcanza **97.4% de exactitud en test** con una pérdida de
**0.0854 en test**: la red aprende de verdad, sin frameworks.

**¿Qué entendí ahora que antes no entendía?**

- **Backpropagation no es magia: es la regla de la cadena multiplicando derivadas hacia atrás.**
  Cada capa guarda lo mínimo (su entrada del forward) y sus gradientes se reducen a un par de
  transposiciones y productos punto.
- **La combinación softmax + cross-entropy es un regalo matemático**: al derivarlas juntas, los
  términos se cancelan y el gradiente queda en la simple resta $p - y$.
- **El descenso por gradiente con lotes chicos es ruidoso pero barato**: cada paso usa solo 64
  imágenes; el ruido incluso ayuda a escapar de mínimos malos, y converge en 10 épocas sin
  ajustar la tasa de aprendizaje.
- **La normalización y la inicialización deciden entre aprender y no aprender**: sin dividir por
  255 los píxeles saturan la señal, y con pesos mal inicializados las neuronas mueren o explotan
  antes de la primera época.
- **El estancamiento de la validación delata el sobreentrenamiento**: en la última época train
  sigue bajando (pérdida 0.0580 → 0.0551, exactitud 98.49% → 98.54%) mientras la validación se
  plancha (exactitud 97.40% en ambas, pérdida 0.0902 → 0.0899); cuando la curva de train sigue
  mejorando y la de validación deja de hacerlo, esa es la señal que marcaría un early stopping.

**¿Qué dígito confunde más la red?** De los 262 errores del test, el par dominante es
**4 → 9** con 21 errores, más del doble que cualquier otro; detrás, empatados con 10 cada uno,
van **7 → 9**, **5 → 3**, **7 → 1** y **2 → 8**. Son confusiones de forma, no de dibujo: un 4
cerrado arriba comparte su silueta con el rulo del 9, el 7 según su inclinación se acerca a un
9 o a un 1, el 3 y el 5 comparten el cuerpo redondeado, y el 2 y el 8 tienen los mismos bucles.
A la resolución de 28x28 la red no resuelve los matices finos: agrupa dibujos que comparten la
misma estructura general. La sección 4 muestra los ejemplos concretos de aciertos y de errores.